# SAXS Dataset Visualizer

Pick a dataset from the `smartt.data_containers` registry, load whichever of
`main` / `remount` / `combined` DataContainers it has, and browse the raw
projections interactively. Also reports basic statistics (range, mean/std,
percentiles) and histograms of the projection intensities and diode values.

Change `DATASET` in the configuration cell below and re-run from there
downstream to switch datasets.

In [1]:
import sys
sys.path.insert(0, '/myhome/smartt')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

from smartt.data_containers import get_dataset, REGISTRY

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


## Configuration

In [ ]:
print('Available datasets:', sorted(REGISTRY))

DATASET = 'cf-peek'   # <- change me, then re-run from the next cell down

Available datasets: ['b411', 'cf-peek', 'fiber-synthetic', 'fiber-synthetic-full', 'frogbone', 'nielsen-m', 'nielsen-mammoth', 'nielsen-t', 'synthetic-b411', 'zenodo']


## Load dataset

In [4]:
ds = get_dataset(DATASET)
dc_types = ds.available_dc_types()
dcs = {t: ds.get_dc(t) for t in dc_types}

print(f'{DATASET!r} -> dc_types available: {dc_types}\n')
for t, dc in dcs.items():
    n = len(dc.projections)
    print(f'  {t:9s}  n_projections={n:4d}  '
          f'volume_shape={tuple(dc.geometry.volume_shape)}  '
          f'projection_shape={tuple(dc.geometry.projection_shape)}  '
          f'full_circle_covered={dc.geometry.full_circle_covered}')

INFO:Rotation matrix generated from inner and outer angles, along with inner and outer rotation axis vectors. Rotation and tilt angles assumed to be in radians.


/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:228: DeprecationWarning: Entry name rotations is deprecated. Use inner_angle instead.
  _deprecated_key_warning('rotations')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:237: DeprecationWarning: Entry name tilts is deprecated. Use outer_angle instead.
  _deprecated_key_warning('tilts')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:269: DeprecationWarning: Entry name offset_j is deprecated. Use j_offset instead.
  _deprecated_key_warning('offset_j')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:279: DeprecationWarning: Entry name offset_k is deprecated. Use k_offset instead.
  _deprecated_key_warning('offset_k')


INFO:No sample geometry information was found. Default mumott geometry assumed.
INFO:No detector geometry information was found. Default mumott geometry assumed.
'nielsen-mammoth' -> dc_types available: ['main']

  main       n_projections= 417  volume_shape=(np.int64(60), np.int64(60), np.int64(80))  projection_shape=(np.int64(60), np.int64(80))  full_circle_covered=False


## Projection viewer

Scroll through projections (and detector-angle channels) for any of the
available `dc_type`s. `idx`/`channel` sliders are shared across `dc_type`s and
clamped to whatever range the selected one actually has (smaller datasets
just stop scrolling early — noted in the title when clamped).

In [5]:
_max_idx = max(len(dc.projections) for dc in dcs.values()) - 1
_max_ch  = max(dc.projections[0].data.shape[-1] for dc in dcs.values()) - 1


@interact(
    dc_type=widgets.Dropdown(options=dc_types, description='dc_type'),
    idx=widgets.IntSlider(min=0, max=_max_idx, step=1, continuous_update=False, description='projection'),
    channel=widgets.IntSlider(min=0, max=_max_ch, step=1, continuous_update=False, description='channel'),
)
def _view_projection(dc_type, idx, channel):
    dc = dcs[dc_type]
    n = len(dc.projections)
    i = min(idx, n - 1)
    p = dc.projections[i]
    m = p.data.shape[-1]
    c = min(channel, m - 1)

    inner_deg = np.degrees(p.inner_angle)
    outer_deg = np.degrees(p.outer_angle)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    vmax = np.percentile(p.data, 99)

    im0 = axes[0].imshow(p.diode.T, origin='lower', cmap='inferno')
    axes[0].set_title('diode')
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(p.data[..., c].T, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
    axes[1].set_title(f'channel {c}/{m - 1}')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    note = '' if (i == idx and c == channel) else '  (clamped to range)'
    fig.suptitle(f'{DATASET} / {dc_type}   proj={i}/{n - 1}   '
                 f'phi={inner_deg:.1f}°   alpha={outer_deg:.1f}°{note}')
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), IntSlider(value=0, con…

### All channels at once

Quick overview of every detector-angle channel for a single projection —
handy for spotting angle-dependent artifacts.

In [ ]:
def plot_all_channels(dc, idx, label=''):
    p = dc.projections[idx]
    n_ch = p.data.shape[-1]
    fig, axes = plt.subplots(1, n_ch + 1, figsize=(2.4 * (n_ch + 1), 3))

    im = axes[0].imshow(p.diode.T, origin='lower', cmap='inferno')
    axes[0].set_title('diode')
    plt.colorbar(im, ax=axes[0], fraction=0.046)

    vmax = np.percentile(p.data, 99)
    for c, ax in enumerate(axes[1:]):
        im = ax.imshow(p.data[..., c].T, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        ax.set_title(f'ch {c}')
        plt.colorbar(im, ax=ax, fraction=0.046)

    inner_deg = np.degrees(p.inner_angle)
    outer_deg = np.degrees(p.outer_angle)
    fig.suptitle(f'{label}  proj={idx}  phi={inner_deg:.1f}°  alpha={outer_deg:.1f}°', y=1.02)
    plt.tight_layout()
    plt.show()


@interact(
    dc_type=widgets.Dropdown(options=dc_types, description='dc_type'),
    idx=widgets.IntSlider(min=0, max=_max_idx, step=1, continuous_update=False, description='projection'),
)
def _view_all_channels(dc_type, idx):
    dc = dcs[dc_type]
    i = min(idx, len(dc.projections) - 1)
    plot_all_channels(dc, i, label=f'{DATASET} / {dc_type}')

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), IntSlider(value=0, con…

## Statistics

In [7]:
def projection_stats(dc, sample_stride=1):
    data = dc.data[::sample_stride]
    return {
        'shape':         data.shape,
        'dtype':         data.dtype,
        'min':           float(data.min()),
        'max':           float(data.max()),
        'mean':          float(data.mean()),
        'std':           float(data.std()),
        'p01':           float(np.percentile(data, 1)),
        'p50':           float(np.percentile(data, 50)),
        'p99':           float(np.percentile(data, 99)),
        'frac_zero':     float(np.mean(data == 0)),
        'frac_negative': float(np.mean(data < 0)),
    }


print(f'{"dc_type":9s} {"shape":20s} {"dtype":8s} {"min":>10s} {"max":>10s} '
      f'{"mean":>10s} {"std":>10s} {"p01":>10s} {"p99":>10s} {"%zero":>7s}')
for t, dc in dcs.items():
    s = projection_stats(dc)
    print(f'{t:9s} {str(s["shape"]):20s} {str(s["dtype"]):8s} '
          f'{s["min"]:10.3g} {s["max"]:10.3g} {s["mean"]:10.3g} {s["std"]:10.3g} '
          f'{s["p01"]:10.3g} {s["p99"]:10.3g} {100 * s["frac_zero"]:6.2f}%')

dc_type   shape                dtype           min        max       mean        std        p01        p99   %zero
main      (417, 60, 80, 8)     float64           0        144        8.4       16.3          0       66.1   1.06%


## Histograms

Subsamples projections (roughly 50 evenly spaced ones) so this stays fast on
large datasets.

In [9]:
@interact(dc_type=widgets.Dropdown(options=dc_types, description='dc_type'))
def _view_histograms(dc_type):
    dc = dcs[dc_type]
    stride = max(1, len(dc.projections) // 50)
    data = dc.data[::stride]
    has_diode = hasattr(dc, 'diode')

    fig, axes = plt.subplots(1, 2 if has_diode else 1, figsize=(10, 3.5))
    axes = np.atleast_1d(axes)

    vals = data.ravel()
    axes[0].hist(vals[vals > 0], bins=100, log=True, color='steelblue')
    axes[0].set_xlabel('SAXS intensity')
    axes[0].set_ylabel('count (log)')
    axes[0].set_title(f'{dc_type}: data distribution (every {stride}th proj.)')

    if has_diode:
        diode = dc.diode[::stride]
        axes[1].hist(diode.ravel(), bins=80, color='darkorange')
        axes[1].set_xlabel('diode (flux monitor)')
        axes[1].set_title(f'{dc_type}: diode distribution')

    plt.tight_layout()
    plt.show()

    s = projection_stats(dc, sample_stride=stride)
    print(f'range=[{s["min"]:.4g}, {s["max"]:.4g}]   mean={s["mean"]:.4g}   std={s["std"]:.4g}   '
          f'p01={s["p01"]:.4g}   p50={s["p50"]:.4g}   p99={s["p99"]:.4g}   '
          f'frac_zero={100 * s["frac_zero"]:.2f}%')

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), Output()), _dom_classe…